# Clase 08 — Manejo de Errores: Excepciones

## Objetivos de la clase

Al terminar esta clase vas a poder:

- Explicar la diferencia entre un **error** (grave, del entorno/sistema) y una **excepción** (recuperable, generada por tu propio código en ejecución).
- Usar `try`/`except` para evitar que tu programa se detenga cuando algo sale mal.
- Capturar **excepciones específicas** (`ZeroDivisionError`, `ValueError`, `IndexError`, `KeyError`) en lugar de usar un `except` genérico.
- Combinar varias cláusulas `except`, agrupar excepciones en una tupla, y usar un `except` genérico como red de respaldo al final.
- Usar `else` (código que corre solo si no hubo excepción) y `finally` (código que corre siempre).
- Escribir funciones robustas que manejan errores de entrada sin romper el programa.


## ¿Por qué te importa esto?

Hasta ahora escribiste código asumiendo que todo sale bien. En la vida real de la ciencia de datos eso casi nunca pasa: archivos que no están donde esperabas, columnas de texto con un `"N/A"` colado en medio de números, diccionarios a los que les falta una clave porque el dataset cambió de formato.

Sin manejo de excepciones, cualquiera de esos problemas tira abajo tu programa entero. Con manejo de excepciones, decides tú: saltas esa fila, avisas al usuario, usas un valor por defecto — pero el programa sigue vivo.


## 1. Errores vs. excepciones: no es lo mismo

Un **error** es un problema serio que viene del entorno, del hardware o de fallos graves de sintaxis. Suele ser tan severo que el programa no puede seguir funcionando; normalmente no lo "manejas" en tiempo de ejecución, lo corriges antes de correr el código (por ejemplo, un `SyntaxError`).

Una **excepción** surge de la ejecución de tu código en una situación concreta: dividiste por cero, convertiste un texto no numérico, pediste una posición que no existe. **Sí las puedes capturar y manejar** con `try`/`except`, dejando que el programa siga adelante.

| Aspecto | Errores | Excepciones |
|---|---|---|
| Origen | Entorno, hardware, sistema | Ejecución problemática de tu código |
| Gravedad | Graves, tumban el programa | Menos graves, se pueden manejar |
| Manejo | No se capturan con `try`/`except` | Se capturan con `try`/`except` |
| Ejemplos | `SyntaxError`, fallos de configuración | `ZeroDivisionError`, `ValueError`, `KeyError` |


In [ ]:
# Una excepción: Python puede levantarla en tiempo de ejecución,
# pero el código en sí es sintácticamente correcto.
try:
    resultado = 10 / 0
except ZeroDivisionError as e:
    print("Se lanzó una excepción:", type(e).__name__, "->", e)

# Un error de sintaxis ni siquiera llega a ejecutarse; Python se
# quejaría antes de correr el programa, y por eso nunca se "atrapa"
# con try/except -- se corrige directamente en el código. Por ejemplo
# (no se ejecuta, solo para ilustrar):
#     def calcular(x
#         return x * 2
#     SyntaxError: '(' was never closed


## 2. `try`/`except`: la red de seguridad básica

El código que puede fallar va dentro del bloque `try`. Si no pasa nada raro, Python lo ejecuta completo y sigue de largo, ignorando el `except`. Si ocurre una excepción, Python interrumpe el `try` justo donde falló y salta directo al `except` correspondiente. Después, el programa continúa con normalidad — no se detiene.


In [ ]:
a = 1
b = 0  # cambia este valor (por ejemplo a 5) y vuelve a correr la celda

try:
    resultado = a / b
    print("Éxito, resultado =", resultado)
except ZeroDivisionError:
    print("Error: no se puede dividir por cero")

# Esta línea se ejecuta pase lo que pase dentro del try/except
print("El programa sigue funcionando con normalidad")


Puedes escribir un `except` sin especificar ningún tipo de excepción (un **except genérico**), y capturará cualquier cosa que salga mal dentro del `try`. Funciona, pero como vas a ver enseguida, esconde información valiosa.


In [ ]:
valores_a_probar = ["5", "abc"]

for texto in valores_a_probar:
    try:
        numero = int(texto)
        resultado = 10 / numero
        print("Resultado:", resultado)
    except:
        print(f"Algo salió mal al procesar {texto!r}")


## 3. Excepciones específicas: por qué el `except` genérico te esconde problemas

El problema de `except:` a secas es doble: (1) oculta información — si tu código falla por un bug real que no anticipaste, el `except` genérico lo traga en silencio igual que a las excepciones "normales", y te quedas sin pistas; (2) te impide reaccionar distinto según el problema — no es lo mismo que el usuario haya escrito texto (`ValueError`) a que haya dividido por cero (`ZeroDivisionError`).

La solución es capturar tipos específicos de excepción, cada uno con su propio bloque `except`:


In [ ]:
def dividir(a, texto_b):
    try:
        b = int(texto_b)
        resultado = a / b
        print(f"Éxito, resultado = {resultado}")
    except ZeroDivisionError:
        print("El número que diste no puede dividir porque es cero")
    except ValueError:
        print("No ingresaste un número válido")

dividir(10, "2")
dividir(10, "0")
dividir(10, "abc")


También puedes agrupar varios tipos en una sola cláusula si quieres darles el mismo tratamiento, usando una tupla:


In [ ]:
try:
    resultado = 1 / 0
except (ZeroDivisionError, NameError):
    # se ejecuta si ocurre cualquiera de estos dos tipos
    print("Ocurrió un problema numérico o de variable")


Una buena práctica es combinar cláusulas específicas con un `except` genérico **al final**, como red de respaldo para lo que no anticipaste — pero nunca como tu única línea de defensa. Python evalúa los `except` en orden, de arriba hacia abajo, y ejecuta el primero que coincida con el tipo de excepción lanzada.


In [ ]:
def dividir_seguro(a, texto_b):
    try:
        b = int(texto_b)
        resultado = a / b
        print(f"Éxito, resultado = {resultado}")
    except ZeroDivisionError:
        print("El número que diste no puede dividir porque es cero")
    except ValueError:
        print("No ingresaste un número válido")
    except Exception:
        print("Algo salió mal que no esperábamos")

dividir_seguro(10, "5")
dividir_seguro(10, "0")
dividir_seguro(10, "diez")
dividir_seguro(10, None)   # int(None) es un TypeError -> cae en el except genérico


## 4. `else` y `finally`: código que depende (o no) del resultado

`else` ejecuta código únicamente cuando el bloque `try` terminó **sin ninguna excepción** — útil para separar "lo que puede fallar" de "lo que quiero hacer solo si no falló nada". `finally` va un paso más allá: se ejecuta **siempre**, haya habido excepción o no, y se usa típicamente para tareas de limpieza (cerrar un archivo, avisar que terminó el procesamiento).

El orden es siempre: `try`, uno o más `except`, opcionalmente `else`, y opcionalmente `finally` al final.


In [ ]:
def dividir_completo(a, texto_b):
    try:
        b = int(texto_b)
        resultado = a / b
    except ZeroDivisionError:
        print("El número que diste no puede dividir porque es cero")
    except ValueError:
        print("No ingresaste un número válido")
    else:
        # Solo se ejecuta si no hubo ninguna excepción
        print("Éxito, resultado =", resultado)
    finally:
        # Se ejecuta siempre, haya habido excepción o no
        print("Procesamiento completado")
    print()  # línea en blanco para separar cada prueba

dividir_completo(10, "5")
dividir_completo(10, "0")
dividir_completo(10, "diez")


## 5. Tour por las excepciones más comunes trabajando con datos

Ya viste `ZeroDivisionError` (dividir por cero) y `ValueError` (valor inapropiado, como convertir texto no numérico). Las otras dos que te vas a topar todo el tiempo trabajando con listas y diccionarios son:

- **`IndexError`** — pediste una posición de una lista fuera de su rango válido.
- **`KeyError`** — pediste una clave de un diccionario que no existe. Nota que `.get()` evita este error devolviendo `None` (o un valor por defecto) en lugar de lanzar la excepción.

(Existen muchas más: `FileNotFoundError`, `TypeError`, `AttributeError`, `ImportError`, entre otras — reconocer el nombre en un traceback te dice de inmediato qué salió mal.)


In [ ]:
inventario = ["manzana", "pera", "uva"]
notas = {"ana": 9, "beto": 7}

try:
    print(inventario[5])
except IndexError as e:
    print("IndexError:", e)

try:
    print(notas["carla"])
except KeyError as e:
    print("KeyError:", e)

# .get() evita el KeyError devolviendo un valor por defecto
print("Con .get():", notas.get("carla", "sin nota registrada"))


## 6. Ejercicios resueltos: la misma lógica, como funciones reutilizables

### Ejercicio 1 — `safe_divide`

Una función que divide dos números y maneja `ZeroDivisionError` devolviendo `None` en vez de romper el programa.


In [ ]:
def safe_divide(numerator, denominator):
    try:
        result = numerator / denominator
        return result
    except ZeroDivisionError:
        print("Error: Cannot divide by zero.")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(7, -2))


### Ejercicio 2 — `calcular_raiz`

Una función que calcula la raíz cuadrada con `math.sqrt()` y distingue **dos** motivos distintos de `ValueError`: un texto no numérico (falla la conversión con `float()`) y un número negativo (falla `math.sqrt()`). Usa un `try` anidado dentro del `except` para diferenciarlos, y un `finally` que siempre avisa que terminó.


In [ ]:
import math

def calcular_raiz(entrada):
    try:
        numero = float(entrada)
        resultado = math.sqrt(numero)
        print(f"La raíz cuadrada de {numero} es {resultado}")
    except ValueError:
        try:
            float(entrada)
            # si la conversión sí funciona, el problema fue el número negativo
            print("Entrada inválida: ingresa un número positivo")
        except ValueError:
            # si la conversión también falla, el texto no era numérico
            print("Entrada inválida: no ingresaste un número")
    finally:
        print("Cálculo terminado")

for entrada in ["16", "-4", "diez"]:
    calcular_raiz(entrada)
    print()


### Ejercicio 3 — `complex_calculation`

Una función que calcula `num / (num - 5)` y captura **cualquier** excepción con `except Exception as e`, sin importar de qué tipo sea.


In [ ]:
def complex_calculation(num):
    try:
        result = num / (num - 5)
        print(f"Result: {result}")
    except Exception as e:
        print("An error occurred during calculation.")

complex_calculation(10)   # funciona sin problema
complex_calculation(5)    # num - 5 == 0 -> ZeroDivisionError
complex_calculation("abc")  # "abc" - 5 -> TypeError; el except genérico igual lo atrapa


## Resumen: lo que te llevas de esta clase

- Un **error** es grave y viene del entorno (`SyntaxError`); no se maneja con `try`/`except`. Una **excepción** surge de tu código en ejecución y sí se captura y maneja.
- `try` contiene el código que puede fallar; si ocurre una excepción, Python salta al `except` correspondiente y el programa continúa.
- Capturar tipos específicos (`except ZeroDivisionError:`, `except ValueError:`) es mejor práctica que un `except:` genérico: te permite reaccionar distinto a cada problema y no esconde bugs que no anticipaste.
- Puedes agrupar varios tipos en una tupla `except (TipoA, TipoB):`, y combinar cláusulas específicas con un genérico **al final** como red de respaldo.
- `else` corre solo si el `try` terminó sin errores; `finally` corre siempre, ideal para tareas de limpieza.
- Las excepciones más comunes trabajando con datos: `ZeroDivisionError`, `ValueError`, `IndexError`, `KeyError` (y también `FileNotFoundError`, `TypeError`, `AttributeError`, `ImportError`).
- `.get()` en diccionarios evita un `KeyError` sin necesidad de `try`/`except`.
